## Завдання 1. Setup і typed-схема Olist

In [1]:
!pip install -q pgserver==0.1.4 \
    --python-version 3.12 \
    --only-binary=:all: \
    --target=/usr/local/lib/python3.13/dist-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 12.2 MB/s eta 0:00:00


In [2]:
!pip install -q psycopg2-binary sqlalchemy pandas kagglehub

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw4_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 35.9 MB/s eta 0:00:00
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [3]:
# Dataset import
import kagglehub

dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print(dataset_dir)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
/kaggle/input/brazilian-ecommerce


In [4]:
import os

sorted(os.listdir(dataset_dir))

['olist_customers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_order_payments_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_products_dataset.csv',
 'olist_sellers_dataset.csv',
 'product_category_name_translation.csv']

In [5]:
files = [
    'olist_customers_dataset.csv',
    'olist_orders_dataset.csv',
    'olist_order_items_dataset.csv',
    'olist_products_dataset.csv',
    'olist_sellers_dataset.csv',
    'olist_order_reviews_dataset.csv',
]

for file_name in files:
    df_tmp = pd.read_csv(os.path.join(dataset_dir, file_name))
    print(f"\n{file_name}")
    print("shape:", df_tmp.shape)
    print("columns:", df_tmp.columns.tolist())


olist_customers_dataset.csv
shape: (99441, 5)
columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

olist_orders_dataset.csv
shape: (99441, 8)
columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

olist_order_items_dataset.csv
shape: (112650, 7)
columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

olist_products_dataset.csv
shape: (32951, 9)
columns: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

olist_sellers_dataset.csv
shape: (3095, 4)
columns: ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

olist_order_reviews_dataset.csv
shape: (99224, 7)
columns

In [6]:
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
    'olist_order_reviews_raw': 'olist_order_reviews_dataset.csv',
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)

    df.to_sql(
        table_name,
        engine,
        if_exists='replace',
        index=False,
        chunksize=10_000
    )

    print(table_name, df.shape)

olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)
olist_order_reviews_raw (99224, 7)


In [7]:
with engine.begin() as conn:
    conn.execute(text("""

      DROP TABLE IF EXISTS customer_segments    CASCADE;
      DROP TABLE IF EXISTS seller_score         CASCADE;
      DROP TABLE IF EXISTS seller_alerts        CASCADE;
      DROP TABLE IF EXISTS olist_customer_dim   CASCADE;
      DROP TABLE IF EXISTS olist_order_reviews  CASCADE;
      DROP TABLE IF EXISTS olist_order_items    CASCADE;
      DROP TABLE IF EXISTS olist_orders         CASCADE;
      DROP TABLE IF EXISTS olist_customers      CASCADE;
      DROP TABLE IF EXISTS olist_products       CASCADE;
      DROP TABLE IF EXISTS olist_sellers        CASCADE;

      CREATE TABLE olist_customers (
          customer_id              TEXT PRIMARY KEY,
          customer_unique_id       TEXT NOT NULL,
          customer_zip_code_prefix INTEGER,
          customer_city            TEXT,
          customer_state           CHAR(2) NOT NULL
      );

      CREATE INDEX idx_olist_customers_unique_id
          ON olist_customers (customer_unique_id);

      CREATE TABLE olist_orders (
          order_id                      TEXT PRIMARY KEY,
          customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
          order_status                  TEXT NOT NULL,
          order_purchase_timestamp      TIMESTAMP NOT NULL,
          order_approved_at             TIMESTAMP,
          order_delivered_carrier_date  TIMESTAMP,
          order_delivered_customer_date TIMESTAMP,
          order_estimated_delivery_date TIMESTAMP,
          CHECK (order_status IN (
              'created', 'approved', 'invoiced', 'processing',
              'shipped', 'delivered', 'unavailable', 'canceled'
          ))
      );

      CREATE TABLE olist_products (
          product_id                 TEXT PRIMARY KEY,
          product_category_name      TEXT,
          product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
          product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
          product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
          product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
          product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
          product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
          product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
      );

      CREATE TABLE olist_sellers (
          seller_id              TEXT PRIMARY KEY,
          seller_zip_code_prefix INTEGER,
          seller_city            TEXT,
          seller_state           CHAR(2) NOT NULL
      );

      CREATE TABLE olist_order_items (
          order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
          order_item_id       INTEGER NOT NULL,
          product_id          TEXT REFERENCES olist_products(product_id),
          seller_id           TEXT REFERENCES olist_sellers(seller_id),
          shipping_limit_date TIMESTAMP,
          price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
          freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
          PRIMARY KEY (order_id, order_item_id)
      );

      CREATE INDEX idx_olist_order_items_seller_order
          ON olist_order_items (seller_id, order_id);

      CREATE TABLE olist_order_reviews (
          review_row_id            BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
          review_id                TEXT,
          order_id                 TEXT NOT NULL REFERENCES olist_orders(order_id),
          review_score             SMALLINT CHECK (review_score BETWEEN 1 AND 5),
          review_comment_title     TEXT,
          review_comment_message   TEXT,
          review_creation_date     TIMESTAMP,
          review_answer_timestamp  TIMESTAMP
      );

      CREATE INDEX idx_olist_order_reviews_order_id
          ON olist_order_reviews (order_id);

    """))
print("Successfully created typed tables.")

Successfully created typed tables.


In [8]:
with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO olist_customers
        SELECT
            customer_id,
            customer_unique_id,
            customer_zip_code_prefix::INTEGER,
            customer_city,
            customer_state::CHAR(2)
        FROM olist_customers_raw;

        INSERT INTO olist_orders
        SELECT
            order_id,
            customer_id,
            order_status,
            order_purchase_timestamp::TIMESTAMP,
            order_approved_at::TIMESTAMP,
            order_delivered_carrier_date::TIMESTAMP,
            order_delivered_customer_date::TIMESTAMP,
            order_estimated_delivery_date::TIMESTAMP
        FROM olist_orders_raw;

        INSERT INTO olist_products
        SELECT
            product_id,
            product_category_name,
            product_name_lenght::INTEGER,
            product_description_lenght::INTEGER,
            product_photos_qty::INTEGER,
            product_weight_g::INTEGER,
            product_length_cm::INTEGER,
            product_height_cm::INTEGER,
            product_width_cm::INTEGER
        FROM olist_products_raw;

        INSERT INTO olist_sellers
        SELECT
            seller_id,
            seller_zip_code_prefix::INTEGER,
            seller_city,
            seller_state::CHAR(2)
        FROM olist_sellers_raw;

        INSERT INTO olist_order_items
        SELECT
            order_id,
            order_item_id::INTEGER,
            product_id,
            seller_id,
            shipping_limit_date::TIMESTAMP,
            price::NUMERIC(12, 2),
            freight_value::NUMERIC(12, 2)
        FROM olist_order_items_raw;

        INSERT INTO olist_order_reviews (
            review_id,
            order_id,
            review_score,
            review_comment_title,
            review_comment_message,
            review_creation_date,
            review_answer_timestamp
        )
        SELECT
            review_id,
            order_id,
            review_score::SMALLINT,
            review_comment_title,
            review_comment_message,
            review_creation_date::TIMESTAMP,
            review_answer_timestamp::TIMESTAMP
        FROM olist_order_reviews_raw;
    """))

print("Raw data successfully loaded into typed tables.")

Raw data successfully loaded into typed tables.


In [9]:
pd.read_sql("""
    SELECT 'customers' AS table_name, COUNT(*) AS n_rows
    FROM olist_customers

    UNION ALL

    SELECT 'orders', COUNT(*)
    FROM olist_orders

    UNION ALL

    SELECT 'order_items', COUNT(*)
    FROM olist_order_items

    UNION ALL

    SELECT 'products', COUNT(*)
    FROM olist_products

    UNION ALL

    SELECT 'sellers', COUNT(*)
    FROM olist_sellers

    UNION ALL

    SELECT 'reviews', COUNT(*)
    FROM olist_order_reviews

    ORDER BY table_name;
""", engine)

,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,reviews,99224
5,sellers,3095


## Завдання 2. DML-запити з PostgreSQL-фічами

### 2.1. INSERT ... RETURNING: створення alert для продавця з найбільшою кількістю запізнень

In [10]:
with engine.begin() as conn:
    # Create the alerts table
    conn.execute(text("""
        DROP TABLE IF EXISTS seller_alerts CASCADE;

        CREATE TABLE seller_alerts (
            alert_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
            seller_id  TEXT NOT NULL REFERENCES olist_sellers(seller_id),
            alert_type TEXT NOT NULL
                CHECK (alert_type IN ('late_delivery', 'low_rating', 'data_quality')),
            severity   SMALLINT NOT NULL
                CHECK (severity BETWEEN 1 AND 5),
            details    TEXT,
            created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
        );
    """))

    # Find the seller with the most late deliveries and create an alert
    result = conn.execute(text("""
        WITH seller_orders AS (
            SELECT DISTINCT
                oi.seller_id,
                oi.order_id
            FROM olist_order_items AS oi
        ),
        late_sellers AS (
            SELECT
                so.seller_id,
                COUNT(DISTINCT o.order_id) AS late_orders
            FROM seller_orders AS so
            JOIN olist_orders AS o
                ON o.order_id = so.order_id
            WHERE o.order_delivered_customer_date IS NOT NULL
              AND o.order_estimated_delivery_date IS NOT NULL
              AND o.order_delivered_customer_date::DATE
                    > o.order_estimated_delivery_date::DATE
            GROUP BY so.seller_id
            ORDER BY late_orders DESC, so.seller_id
            LIMIT 1
        )
        INSERT INTO seller_alerts (
            seller_id,
            alert_type,
            severity,
            details
        )
        SELECT
            seller_id,
            'late_delivery',
            4,
            'Seller has ' || late_orders ||
            ' late delivered orders in the loaded dataset'
        FROM late_sellers
        RETURNING
            alert_id,
            seller_id,
            alert_type,
            severity,
            details,
            created_at;
    """))

    rows = result.fetchall()
    columns = result.keys()

pd.DataFrame(rows, columns=columns)

,alert_id,seller_id,alert_type,severity,details,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,Seller has 172 late delivered orders in the lo...,2026-10-05 11:51:16.726486+00:00


Запит створив alert типу `late_delivery` для продавця з найбільшою кількістю замовлень із простроченою доставкою. У цього продавця було 172 прострочені доставки. Використання `RETURNING` одразу повернуло доданий рядок, включно зі згенерованими `alert_id` і `timestamp`, що робить цю операцію зручною для перевірки та аудиту в notebook.

### 2.2. INSERT ... ON CONFLICT DO UPDATE: idempotent upsert агрегованих seller-score

In [11]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS seller_score CASCADE;

        CREATE TABLE seller_score (
            seller_id   TEXT PRIMARY KEY
                REFERENCES olist_sellers(seller_id),
            avg_rating  NUMERIC(3, 2),
            n_reviews   INTEGER NOT NULL DEFAULT 0
                CHECK (n_reviews >= 0),
            tier        TEXT
                CHECK (tier IN ('gold', 'silver', 'bronze')),
            updated_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
        );
    """))

    result = conn.execute(text("""
        WITH seller_orders AS (
            SELECT DISTINCT
                oi.seller_id,
                oi.order_id
            FROM olist_order_items AS oi
        ),

        review_per_order AS (
            SELECT
                rv.order_id,
                AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
            FROM olist_order_reviews AS rv
            WHERE rv.review_score IS NOT NULL
            GROUP BY rv.order_id
        ),

        seller_rating AS (
            SELECT
                so.seller_id,
                AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
                COUNT(rpo.order_id) AS n_reviews
            FROM seller_orders AS so
            LEFT JOIN review_per_order AS rpo
                ON rpo.order_id = so.order_id
            GROUP BY so.seller_id
        )

        INSERT INTO seller_score (
            seller_id,
            avg_rating,
            n_reviews,
            tier
        )
        SELECT
            seller_id,
            avg_rating,
            n_reviews,
            CASE
                WHEN avg_rating IS NULL THEN NULL
                WHEN avg_rating >= 4.5 THEN 'gold'
                WHEN avg_rating >= 3.5 THEN 'silver'
                ELSE 'bronze'
            END AS tier
        FROM seller_rating

        ON CONFLICT (seller_id) DO UPDATE
        SET
            avg_rating = EXCLUDED.avg_rating,
            n_reviews  = EXCLUDED.n_reviews,
            tier       = EXCLUDED.tier,
            updated_at = NOW()

        RETURNING
            seller_id,
            avg_rating,
            n_reviews,
            tier,
            updated_at;
    """))

    rows = result.fetchall()
    columns = result.keys()

seller_score_df = pd.DataFrame(rows, columns=columns)

seller_score_df.head(10)

,seller_id,avg_rating,n_reviews,tier,updated_at
0,a61cc04793308395a840807104365121,5.00,1,gold,2026-10-05 11:52:59.041218+00:00
1,6d2f2e3b539480db1e0842b3a4e32e6e,3.67,3,silver,2026-10-05 11:52:59.041218+00:00
2,c53bcd3be457a342a97e39e5a9f0be22,3.20,5,bronze,2026-10-05 11:52:59.041218+00:00
3,9591fc341b1bfb7ef561e2968ec6e011,1.00,1,bronze,2026-10-05 11:52:59.041218+00:00
4,f9ec7093df3a7b346b7bcf7864069ca3,4.78,9,gold,2026-10-05 11:52:59.041218+00:00
5,fc38b5dceee1a730fad8853453437fbd,4.60,5,gold,2026-10-05 11:52:59.041218+00:00
6,b39d7fe263ef469605dbb32608aee0af,3.88,43,silver,2026-10-05 11:52:59.041218+00:00
7,5def4c3732941a971cba8fdee992ede1,4.60,5,gold,2026-10-05 11:52:59.041218+00:00
8,ea65d8b58316a6f2362f2a9e4b3e86ad,4.00,9,silver,2026-10-05 11:52:59.041218+00:00
9,bf0d50a6410d487dc97d2baac0a8c0be,2.00,1,bronze,2026-10-05 11:52:59.041218+00:00


In [12]:
# Disrtibution by tier
with engine.begin() as conn:
    result = conn.execute(text("""
      SELECT
          COALESCE(tier, 'no_rating') AS tier_group,
          COUNT(*) AS seller_count
      FROM seller_score
      GROUP BY tier
      ORDER BY seller_count DESC;
    """))
    rows = result.fetchall()
    columns = result.keys()

pd.DataFrame(rows, columns=columns)

,tier_group,seller_count
0,silver,1568
1,gold,983
2,bronze,539
3,no_rating,5


Більшість продавців належать до срібного рівня, далі йдуть золотий і бронзовий. Лише 5 продавців не мають рейтингу, що свідчить про досить високий рівень охоплення відгуками серед усіх продавців.

2.3. UPDATE ... FROM: оновлення прапорця запізнення в olist_orders

In [13]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE olist_orders
        ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;
    """))

    result = conn.execute(text("""
        WITH delivery_flags AS (
            SELECT
                order_id,
                (
                    order_delivered_customer_date IS NOT NULL
                    AND order_estimated_delivery_date IS NOT NULL
                    AND order_delivered_customer_date::DATE
                        > order_estimated_delivery_date::DATE
                ) AS is_late_calc
            FROM olist_orders
        )

        UPDATE olist_orders AS o
        SET is_late = f.is_late_calc
        FROM delivery_flags AS f
        WHERE f.order_id = o.order_id
        RETURNING
            o.order_id,
            o.customer_id,
            o.is_late;
    """))

    rows = result.fetchall()
    columns = result.keys()

updated_orders_df = pd.DataFrame(rows, columns=columns)
updated_orders_df.head(10)

,order_id,customer_id,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,False
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,False
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,False
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,False
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,False
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,False


In [14]:
pd.read_sql("""
    SELECT
        is_late,
        COUNT(*) AS n_orders
    FROM olist_orders
    GROUP BY is_late
    ORDER BY is_late;
""", engine)

,is_late,n_orders
0,False,92906
1,True,6535


Прапорець простроченої доставки був перерахований для всіх замовлень на основі дат доставки. Оскільки це значення обчислюється заново, а не змінюється поступово, повторний запуск `UPDATE` дає той самий результат, якщо вихідні дані не змінилися.

### 2.4. DELETE ... RETURNING: контрольоване видалення застарілих alert-ів

In [15]:
with engine.begin() as conn:
    result = conn.execute(text("""
        DELETE FROM seller_alerts
        WHERE created_at < NOW() - INTERVAL '30 days'
        RETURNING
            alert_id,
            seller_id,
            alert_type,
            created_at;
    """))

    rows = result.fetchall()
    columns = result.keys()

deleted_alerts_df = pd.DataFrame(rows, columns=columns)
deleted_alerts_df

,alert_id,seller_id,alert_type,created_at


Жодного `alert` не було видалено, оскільки всі поточні `alert` були створені нещодавно і їм ще не виповнилося 30 днів. Тому для поточного стану цього набору даних порожній результат є очікуваним.

## Завдання 3. CREATE TABLE customer_segments

In [16]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS customer_segments CASCADE;
        DROP TABLE IF EXISTS olist_customer_dim CASCADE;

        CREATE TABLE olist_customer_dim (
            customer_unique_id TEXT PRIMARY KEY,
            latest_customer_id TEXT REFERENCES olist_customers(customer_id),
            customer_state     CHAR(2),
            customer_city      TEXT
        );

        INSERT INTO olist_customer_dim (
            customer_unique_id,
            latest_customer_id,
            customer_state,
            customer_city
        )
        SELECT DISTINCT ON (c.customer_unique_id)
            c.customer_unique_id,
            c.customer_id AS latest_customer_id,
            c.customer_state,
            c.customer_city
        FROM olist_customers AS c
        LEFT JOIN olist_orders AS o
            ON o.customer_id = c.customer_id
        ORDER BY
            c.customer_unique_id,
            o.order_purchase_timestamp DESC NULLS LAST,
            c.customer_id;
    """))

In [17]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE customer_segments (
            segment_id          BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

            customer_unique_id  TEXT NOT NULL,

            segment_name        TEXT NOT NULL
                CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),

            rfm_score           SMALLINT NOT NULL
                CHECK (rfm_score BETWEEN 1 AND 5),

            recency_days        INTEGER
                CHECK (recency_days IS NULL OR recency_days >= 0),

            monetary_value      NUMERIC(12, 2) NOT NULL DEFAULT 0
                CHECK (monetary_value >= 0),

            n_orders            INTEGER NOT NULL DEFAULT 0
                CHECK (n_orders >= 0),

            assigned_at         TIMESTAMPTZ NOT NULL DEFAULT NOW(),

            UNIQUE (customer_unique_id)
        );

        ALTER TABLE customer_segments
            ADD CONSTRAINT fk_customer_segments_customer_unique
            FOREIGN KEY (customer_unique_id)
            REFERENCES olist_customer_dim(customer_unique_id)
            ON DELETE CASCADE;
    """))

    result = conn.execute(text("""
        WITH reference_date AS (
            SELECT
                (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
            FROM olist_orders
        ),

        order_totals AS (
            SELECT
                o.order_id,
                o.customer_id,
                o.order_purchase_timestamp::DATE AS order_date,
                COALESCE(
                    SUM(oi.price + oi.freight_value),
                    0
                )::NUMERIC(12, 2) AS order_total
            FROM olist_orders AS o
            LEFT JOIN olist_order_items AS oi
                ON oi.order_id = o.order_id
            GROUP BY
                o.order_id,
                o.customer_id,
                o.order_purchase_timestamp
        ),

        customer_stats AS (
            SELECT
                c.customer_unique_id,
                COUNT(DISTINCT ot.order_id) AS n_orders,
                COALESCE(
                    SUM(ot.order_total),
                    0
                )::NUMERIC(12, 2) AS monetary_value,
                MAX(ot.order_date) AS last_order_date
            FROM olist_customer_dim AS cd
            JOIN olist_customers AS c
                ON c.customer_unique_id = cd.customer_unique_id
            LEFT JOIN order_totals AS ot
                ON ot.customer_id = c.customer_id
            GROUP BY c.customer_unique_id
        ),

        segmented AS (
            SELECT
                cs.customer_unique_id,
                cs.n_orders,
                cs.monetary_value,

                CASE
                    WHEN cs.last_order_date IS NULL THEN NULL
                    ELSE (
                        rd.as_of_date - cs.last_order_date
                    )::INTEGER
                END AS recency_days,

                CASE
                    WHEN cs.monetary_value >= 1000 THEN 'VIP'

                    WHEN cs.last_order_date IS NOT NULL
                         AND (
                             rd.as_of_date - cs.last_order_date
                         )::INTEGER <= 90
                         AND cs.n_orders = 1
                    THEN 'new'

                    WHEN cs.n_orders >= 2
                         OR cs.monetary_value >= 100
                    THEN 'regular'

                    ELSE 'inactive'
                END AS segment_name,

                CASE
                    WHEN cs.monetary_value >= 1000 THEN 5
                    WHEN cs.monetary_value >= 500  THEN 4
                    WHEN cs.n_orders >= 2          THEN 3
                    WHEN cs.n_orders = 1           THEN 2
                    ELSE 1
                END::SMALLINT AS rfm_score

            FROM customer_stats AS cs
            CROSS JOIN reference_date AS rd
        )

        INSERT INTO customer_segments (
            customer_unique_id,
            segment_name,
            rfm_score,
            recency_days,
            monetary_value,
            n_orders
        )

        SELECT
            customer_unique_id,
            segment_name,
            rfm_score,
            recency_days,
            monetary_value,
            n_orders
        FROM segmented

        ON CONFLICT (customer_unique_id) DO UPDATE
        SET
            segment_name   = EXCLUDED.segment_name,
            rfm_score      = EXCLUDED.rfm_score,
            recency_days   = EXCLUDED.recency_days,
            monetary_value = EXCLUDED.monetary_value,
            n_orders       = EXCLUDED.n_orders,
            assigned_at    = NOW();
    """))

In [18]:
pd.read_sql("""
    SELECT
        segment_name,
        COUNT(*) AS n_customers,
        ROUND(AVG(monetary_value), 2) AS avg_monetary_value,
        ROUND(AVG(n_orders), 2) AS avg_orders
    FROM customer_segments
    GROUP BY segment_name
    ORDER BY n_customers DESC;
""", engine)

,segment_name,n_customers,avg_monetary_value,avg_orders
0,regular,45982,224.57,1.07
1,inactive,40032,59.28,1.00
2,new,8891,140.39,1.00
3,VIP,1191,1591.81,1.11


In [19]:
pd.read_sql("""
    SELECT
        (SELECT COUNT(*) FROM olist_customer_dim) AS customer_dim_rows,
        (SELECT COUNT(*) FROM customer_segments) AS segment_rows;
""", engine)

,customer_dim_rows,segment_rows
0,96096,96096


Сегментацію клієнтів було успішно створено для всіх 96 096 унікальних клієнтів. Найбільшими групами є regular та inactive, тоді як VIP-клієнти становлять значно менший сегмент, але мають набагато вищу середню monetary value. Однакова кількість рядків у `olist_customer_dim` і `customer_segments` підтверджує, що для кожного реального клієнта є один запис із сегментацією.

## Завдання 4. JOIN-запити з business-interpretation

**Business-question #1:** Які категорії товарів і штати продавців приносять найбільший дохід від доставлених замовлень?

In [20]:
pd.read_sql("""
    SELECT
        p.product_category_name,
        s.seller_state,
        COUNT(DISTINCT oi.order_id) AS n_orders,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue
    FROM olist_order_items AS oi
    JOIN olist_orders AS o
        ON o.order_id = oi.order_id
    JOIN olist_products AS p
        ON p.product_id = oi.product_id
    JOIN olist_sellers AS s
        ON s.seller_id = oi.seller_id
    WHERE o.order_status = 'delivered'
    GROUP BY
        p.product_category_name,
        s.seller_state
    ORDER BY revenue DESC NULLS LAST
    LIMIT 10;
""", engine)

,product_category_name,seller_state,n_orders,revenue
0,cama_mesa_banho,SP,8214,1070048.14
1,relogios_presentes,SP,4494,1016168.80
2,beleza_saude,SP,5684,790967.66
3,esporte_lazer,SP,4772,680165.54
4,moveis_decoracao,SP,4597,605650.63
5,cool_stuff,SP,2621,484196.53
6,utilidades_domesticas,SP,4021,471810.66
7,automotivo,SP,3018,438833.22
8,informatica_acessorios,SP,2783,388506.42
9,bebes,SP,1981,339129.74


Найвищий дохід переважно припадає на продавців зі штату SP. Серед найкращих результатів найбільший дохід від доставлених замовлень приносять категорії `cama_mesa_banho` та `relogios_presentes`.

**Business-question #2:** Які ознаки можна створити для кожного реального клієнта?

In [21]:
pd.read_sql("""
    WITH order_totals AS (
        SELECT
            o.order_id,
            c.customer_unique_id,
            SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
        FROM olist_orders AS o
        JOIN olist_customers AS c
            ON c.customer_id = o.customer_id
        LEFT JOIN olist_order_items AS oi
            ON oi.order_id = o.order_id
        GROUP BY
            o.order_id,
            c.customer_unique_id
    ),

    review_per_order AS (
        SELECT
            rv.order_id,
            AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
        FROM olist_order_reviews AS rv
        WHERE rv.review_score IS NOT NULL
        GROUP BY rv.order_id
    ),

    customer_stats AS (
        SELECT
            ot.customer_unique_id,
            COUNT(DISTINCT ot.order_id) AS n_orders,
            COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS total_spend,
            AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_review_score
        FROM order_totals AS ot
        LEFT JOIN review_per_order AS rpo
            ON rpo.order_id = ot.order_id
        GROUP BY ot.customer_unique_id
    )

    SELECT
        cd.customer_unique_id,
        cd.customer_state,
        COALESCE(cs.n_orders, 0) AS n_orders,
        COALESCE(cs.total_spend, 0)::NUMERIC(12, 2) AS total_spend,
        COALESCE(cs.avg_review_score, 0)::NUMERIC(3, 2) AS avg_review_score
    FROM olist_customer_dim AS cd
    LEFT JOIN customer_stats AS cs
        ON cs.customer_unique_id = cd.customer_unique_id
    ORDER BY total_spend DESC NULLS LAST
    LIMIT 20;
""", engine)

,customer_unique_id,customer_state,n_orders,total_spend,avg_review_score
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,1.0
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,5.0
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,1.0
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,5.0
4,459bef486812aa25204be022145caa62,ES,1,6922.21,0.0
5,ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,5.0
6,4007669dec559734d6f53e029e360987,MG,1,6081.54,1.0
7,5d0a2980b292d049061542014e8960bf,GO,1,4809.44,1.0
8,eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,4.0
9,48e1ac109decbb87765a3eade6854098,PB,1,4681.78,5.0


Ознаки на рівні клієнта були успішно створені за `customer_unique_id`. Результат поєднує кількість замовлень, загальну суму витрат і середню оцінку відгуків, при цьому клієнти зберігаються в результаті навіть тоді, коли деякі пов’язані дані відсутні.

**Business-question #3:** Які штати зустрічаються серед клієнтів, продавців або і тих, і інших?

In [22]:
pd.read_sql("""
    WITH customer_states AS (
        SELECT
            customer_state AS state,
            COUNT(*) AS n_customers
        FROM olist_customers
        GROUP BY customer_state
    ),

    seller_states AS (
        SELECT
            seller_state AS state,
            COUNT(*) AS n_sellers
        FROM olist_sellers
        GROUP BY seller_state
    )

    SELECT
        COALESCE(c.state, s.state) AS state,
        c.n_customers,
        s.n_sellers,
        CASE
            WHEN c.state IS NOT NULL
             AND s.state IS NOT NULL THEN 'both'
            WHEN c.state IS NOT NULL THEN 'customers_only'
            ELSE 'sellers_only'
        END AS state_presence
    FROM customer_states AS c
    FULL OUTER JOIN seller_states AS s
        ON s.state = c.state
    ORDER BY state;
""", engine)

,state,n_customers,n_sellers,state_presence
0,AC,81,1.0,both
1,AL,413,NaN,customers_only
2,AM,148,1.0,both
3,AP,68,NaN,customers_only
4,BA,3380,19.0,both
5,CE,1336,13.0,both
6,DF,2140,30.0,both
7,ES,2033,23.0,both
8,GO,2020,40.0,both
9,MA,747,1.0,both


Більшість бразильських штатів у результаті містять і клієнтів, і продавців, але деякі штати, наприклад AL та AP, представлені лише серед клієнтів. FULL OUTER JOIN дозволяє побачити такі випадки, коли штат присутній лише з одного боку.

**Business-question #4:** Які пари продавців знаходяться в одному штаті, але в різних містах?

In [23]:
pd.read_sql("""
    SELECT
        s1.seller_id AS seller_a,
        s2.seller_id AS seller_b,
        s1.seller_state,
        s1.seller_city AS city_a,
        s2.seller_city AS city_b
    FROM olist_sellers AS s1
    JOIN olist_sellers AS s2
        ON s1.seller_state = s2.seller_state
       AND s1.seller_id < s2.seller_id
       AND s1.seller_city IS DISTINCT FROM s2.seller_city
    ORDER BY
        s1.seller_state,
        seller_a,
        seller_b
    LIMIT 20;
""", engine)

,seller_a,seller_b,seller_state,city_a,city_b
0,1444c08e64d55fb3c25f0f09c07ffcf2,2b402d5dc42554061f8ea98d1916f148,BA,lauro de freitas,irece
1,1444c08e64d55fb3c25f0f09c07ffcf2,398cb257329ef7af7f1943a8974a3cbc,BA,lauro de freitas,salvador
2,1444c08e64d55fb3c25f0f09c07ffcf2,4221a7df464f1fe2955934e30ff3a5a1,BA,lauro de freitas,bahia
3,1444c08e64d55fb3c25f0f09c07ffcf2,43753b27d77860f1654aa72e251a7878,BA,lauro de freitas,barro alto
4,1444c08e64d55fb3c25f0f09c07ffcf2,4aba391bc3b88717ce08eb11e44937b2,BA,lauro de freitas,arraial d'ajuda (porto seguro)
5,1444c08e64d55fb3c25f0f09c07ffcf2,4fb41dff7c50136976d1a5cf004a42e2,BA,lauro de freitas,feira de santana
6,1444c08e64d55fb3c25f0f09c07ffcf2,651530bf5c607240ccdd89a30c9c9712,BA,lauro de freitas,ipira
7,1444c08e64d55fb3c25f0f09c07ffcf2,659e8466eb3ff1b0e8740d74fb7bbedd,BA,lauro de freitas,eunapolis
8,1444c08e64d55fb3c25f0f09c07ffcf2,75d34ebb1bd0bd7dde40dd507b8169c3,BA,lauro de freitas,salvador
9,1444c08e64d55fb3c25f0f09c07ffcf2,a3dd39f583bc80bd8c5901c95878921e,BA,lauro de freitas,salvador


Запит знаходить пари різних продавців, які знаходяться в одному штаті, але в різних містах. Перші результати стосуються штату BA, оскільки результат відсортований за штатом і обмежений першими 20 рядками.

**Business-question #5:** У яких замовлень немає запису про відгук?

In [24]:
pd.read_sql("""
    SELECT
        o.order_id,
        o.customer_id,
        o.order_status,
        o.order_purchase_timestamp
    FROM olist_orders AS o
    LEFT JOIN olist_order_reviews AS rv
        ON rv.order_id = o.order_id
    WHERE rv.review_row_id IS NULL
    ORDER BY o.order_purchase_timestamp
    LIMIT 20;
""", engine)

,order_id,customer_id,order_status,order_purchase_timestamp
0,cda873529ca7ab71f677d5ec11a40304,76c74aaff2f3f7355f46d9818ad092b8,shipped,2016-10-05 16:57:30
1,5da2004d85f2828349d887c989da2adc,95f592086150f0c95d3082aea047b816,delivered,2016-10-05 20:20:59
2,02190241f7190a1f3c7e0df95a749c6a,55c9044401ffab0c717f24405e2e042f,delivered,2016-10-07 10:52:22
3,0efd0bc268d34da3f01f4ff25a6d4335,95446917717bb58d553d107d0f1668f6,shipped,2016-10-07 15:53:31
4,c776863a93dc0740c6e7d78104b21413,1710b798ebdc5db6665c815b90a64999,delivered,2016-10-10 15:01:27
5,d730d8d0ea3149120703933fdf98b2d4,ce2a66464da8b80dedd7639a6e489a58,shipped,2017-01-13 20:12:41
6,05cd156f0f47579a36d7a52724fc31b8,976224a95faf328b17654b1b1411d5e1,delivered,2017-01-14 18:27:28
7,719b940ab2f1e351ffb14a0a02ffe3c4,ede9a1daf58d256c0d648ef7e1479010,delivered,2017-01-16 13:01:51
8,fbd82b1fe98967de0db29b6cf134d00a,45f5c9d984dbfa2105e3544cc85133bf,delivered,2017-01-18 08:55:36
9,171398a0cd843c65ac78aee9e05a1f49,f0244902c07b7c0799e468218579a925,delivered,2017-01-18 16:07:37


Запит знаходить замовлення, для яких немає відповідного запису про відгук. Результат містить замовлення з різними статусами, включно з доставленими, що показує: для деяких завершених замовлень клієнти все одно не залишили відгук.

## Завдання 5. GROUP BY, HAVING і агрегати

## 5.1 Які категорії товарів приносять значний дохід?

In [25]:
pd.read_sql("""
    SELECT
        p.product_category_name,
        COUNT(DISTINCT oi.order_id) AS n_orders,
        SUM(oi.price)::NUMERIC(12, 2) AS gross_revenue,
        SUM(oi.freight_value)::NUMERIC(12, 2) AS freight,
        AVG(oi.price)::NUMERIC(8, 2) AS avg_item_price
    FROM olist_order_items AS oi
    JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY p.product_category_name
    HAVING SUM(oi.price) > 10000
    ORDER BY gross_revenue DESC NULLS LAST;
""", engine)

,product_category_name,n_orders,gross_revenue,freight,avg_item_price
0,beleza_saude,8836,1258681.34,182566.73,130.16
1,relogios_presentes,5624,1205005.68,100535.93,201.14
2,cama_mesa_banho,9417,1036988.68,204693.04,93.30
3,esporte_lazer,7720,988048.97,168607.51,114.34
4,informatica_acessorios,6689,911954.32,147318.08,116.51
5,moveis_decoracao,6449,729762.49,172749.30,87.56
6,cool_stuff,3632,635290.85,84039.10,167.36
7,utilidades_domesticas,5884,632248.66,146149.11,90.79
8,automotivo,3897,592720.11,92664.21,139.96
9,ferramentas_jardim,3518,485256.46,98962.75,111.63


beleza_saude має найвищий валовий дохід — приблизно 1,26 млн, далі йдуть `relogios_presentes` та `cama_mesa_banho`. Деякі категорії з високим доходом суттєво відрізняються за середньою ціною товару. Наприклад, pcs має відносно невелику кількість замовлень, але середня ціна товару перевищує 1 000. Наявність `NULL` у категорії товару також вказує на проблему з якістю даних у метаданих товарів.

### 5.2 У яких штатах клієнтів найвища середня вартість замовлення серед штатів із достатньою кількістю замовлень?

In [26]:
pd.read_sql("""
    WITH order_totals AS (
        SELECT
            o.order_id,
            c.customer_state,
            SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
        FROM olist_orders AS o
        JOIN olist_customers AS c
            ON c.customer_id = o.customer_id
        JOIN olist_order_items AS oi
            ON oi.order_id = o.order_id
        GROUP BY
            o.order_id,
            c.customer_state
    )

    SELECT
        customer_state,
        COUNT(*) AS n_orders,
        SUM(order_total)::NUMERIC(12, 2) AS gmv,
        (
            SUM(order_total) / NULLIF(COUNT(*), 0)
        )::NUMERIC(8, 2) AS aov
    FROM order_totals
    GROUP BY customer_state
    HAVING COUNT(*) >= 100
    ORDER BY aov DESC NULLS LAST;
""", engine)

,customer_state,n_orders,gmv,aov
0,PB,532,140987.81,265.01
1,AL,411,96229.40,234.13
2,RO,247,57558.02,233.03
3,PA,970,217647.11,224.38
4,TO,279,61354.42,219.91
5,PI,493,108132.28,219.34
6,SE,345,73032.32,211.69
7,RN,482,101895.08,211.40
8,CE,1327,275606.30,207.69
9,MT,903,186168.96,206.17


PB має найвищу середню вартість замовлення — 265,01 серед штатів, де є щонайменше 100 замовлень. Далі йдуть AL та RO. SP має значно більшу кількість замовлень і GMV, але його середня вартість замовлення нижча — 143,12. Це показує, що високий загальний дохід не обов’язково означає високу середню вартість одного замовлення.

### 5.3 Які категорії товарів купував кожен реальний клієнт?

In [27]:
pd.read_sql("""
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS n_orders,
        SUM(oi.price)::NUMERIC(12, 2) AS total_spend,

        STRING_AGG(
            DISTINCT COALESCE(
                p.product_category_name,
                '(unknown)'
            ),
            ', ' ORDER BY COALESCE(
                p.product_category_name,
                '(unknown)'
            )
        ) AS categories

    FROM olist_customers AS c
    JOIN olist_orders AS o
        ON o.customer_id = c.customer_id
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id

    GROUP BY c.customer_unique_id

    HAVING COUNT(DISTINCT o.order_id) >= 1

    ORDER BY total_spend DESC NULLS LAST
    LIMIT 20;
""", engine)

,customer_unique_id,n_orders,total_spend,categories
0,0a0a92112bd4c708ca5fde585afaa872,1,13440.00,telefonia_fixa
1,da122df9eeddfedc1dc1f5349a1a690c,2,7388.00,eletroportateis
2,763c8b1c9c68a0229c42c9fc6f662b93,1,7160.00,telefonia_fixa
3,dc4802a71eae9be1dd28f5d788ceb526,1,6735.00,utilidades_domesticas
4,459bef486812aa25204be022145caa62,1,6729.00,pcs
5,ff4159b92c40ebe40454e3e6a7c35ed6,1,6499.00,artes
6,4007669dec559734d6f53e029e360987,1,5934.60,agro_industria_e_comercio
7,eebb5dda148d3893cdaf5b5ca3040ccb,1,4690.00,eletroportateis
8,5d0a2980b292d049061542014e8960bf,1,4599.90,cool_stuff
9,48e1ac109decbb87765a3eade6854098,1,4590.00,pcs


Клієнти з найбільшими витратами здебільшого пов’язані з невеликою кількістю дорогих категорій товарів, а багато клієнтів із найвищими витратами мають лише одне замовлення. `STRING_AGG` об’єднує всі унікальні категорії товарів, які купував кожен реальний клієнт, в одну ознаку на рівні клієнта. Для одного з клієнтів із найбільшими витратами категорія має значення (unknown), що підтверджує наявність пропущених значень у категоріях товарів.

## Завдання 6. Set operations, підзапит/window і Reflection

### 6.1 Які штати містять і клієнтів, і продавців?

In [28]:
pd.read_sql("""
    SELECT customer_state AS state
    FROM olist_customers

    INTERSECT

    SELECT seller_state AS state
    FROM olist_sellers

    ORDER BY state;
""", engine)

,state
0,AC
1,AM
2,BA
3,CE
4,DF
5,ES
6,GO
7,MA
8,MG
9,MS


`INTERSECT` повертає лише коди штатів, які присутні і в даних про клієнтів, і в даних про продавців. У цьому випадку він підходить, оскільки питання стосується спільних елементів двох наборів, а не всіх значень з кожного боку.

## Reflection
Найбільш незвичним для мене був `SELF JOIN`, оскільки одна й та сама таблиця використовується у двох логічних ролях, а додаткові умови потрібні для того, щоб уникнути дублювання пар продавців. Anti-join також був корисним, тому що він дає простий спосіб знайти відсутні зв’язки, наприклад замовлення без відгуків.

Найскладнішими для формулювання як бізнес-запитів були ті запити, які стосувалися технічних зв’язків, а не безпосередніх бізнес-метрик. Серед можливостей PostgreSQL, які використовувалися в цьому домашньому завданні, `ON CONFLICT` здається мені особливо корисним для production ETL, оскільки дозволяє повторно виконувати upsert без створення дублікатів. Обмеження (constraints) також важливі, тому що вони не дозволяють некоректним даним потрапляти в типізовану схему.

У реальному ML pipeline я б використовувала window functions для таких задач, як вибір останньої події клієнта, ранжування транзакцій, створення історичних ознак і розрахунок ознак у часі зі збереженням рівня окремих рядків.

У даних Olist також є кілька проблем із якістю даних. Для деяких замовлень немає відгуків, у деяких товарів відсутні значення категорії, а покриття відгуками не є повним. Такі випадки потрібно обробляти явно, а не просто непомітно видаляти.

Для ML-ознак на рівні клієнта `customer_unique_id` є більш підходящим, ніж `customer_id`, оскільки один реальний клієнт може мати кілька `customer_id`, пов’язаних з окремими замовленнями. Аналогічно, перед розрахунком `seller_score` дані потрібно привести до рівня `seller_id × order_id`, щоб один відгук не враховувався кілька разів через JOIN на рівні окремих товарів.